In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import json
from pathlib import Path
import torch
from torch.utils.data import DataLoader
import lightning as L
from utilities import RegressionDataset, CVDataModule
from pytoda.smiles.smiles_language import SMILESTokenizer
from paccmann_predictor.models import MODEL_FACTORY
from cli_cv_paccmann import Module_training_paccmann
from utilities import precision_at_q, ndcg_at_q

In [2]:
params = {}
with open("paccmann_v2_params.json") as fp:
    params.update(json.load(fp))

params["number_of_genes"] = 2083
params["smiles_vocabulary_size"] = 87

In [15]:
params

{'drug_sensitivity_min_max': True,
 'augment_smiles': True,
 'smiles_start_stop_token': True,
 'number_of_genes': 2083,
 'smiles_padding_length': 512,
 'stacked_dense_hidden_sizes': [1024, 512],
 'activation_fn': 'relu',
 'dropout': 0.5,
 'batch_norm': True,
 'filters': [64, 64, 64],
 'molecule_heads': [4, 4, 4, 4],
 'gene_heads': [2, 2, 2, 2],
 'smiles_embedding_size': 16,
 'kernel_sizes': [[3, 16], [5, 16], [11, 16]],
 'smiles_attention_size': 64,
 'gene_attention_size': 1,
 'embed_scale_grad': False,
 'final_activation': True,
 'batch_size': 256,
 'lr': 0.01,
 'optimizer': 'adam',
 'loss_fn': 'mse',
 'epochs': 10,
 'save_model': 25,
 'dataset_device': 'cpu'}

In [3]:
smiles_language = SMILESTokenizer.from_pretrained("smiles_language.pkl")
smiles_language.set_encoding_transforms(
            add_start_and_stop=params.get("add_start_and_stop", True),
            padding=params.get("padding", True),
            padding_length=params.get("smiles_padding_length", None),
        )
    
smiles_language.set_smiles_transforms(
            augment=params.get("augment_smiles", False),
            canonical=params.get("smiles_canonical", True),
            kekulize=params.get("smiles_kekulize", False),
            all_bonds_explicit=params.get("smiles_bonds_explicit", False),
            all_hs_explicit=params.get("smiles_all_hs_explicit", False),
            remove_bonddir=params.get("smiles_remove_bonddir", False),
            remove_chirality=params.get("smiles_remove_chirality", False),
            selfies=params.get("selfies", False),
            sanitize=params.get("selfies", False),
        )

In [4]:
model_name = params.get("model_fn", "paccmann_v2")
model = MODEL_FACTORY[model_name](params)
model._associate_language(smiles_language)


In [5]:
gdsc_gex = pd.read_csv(
    "../../paccmann_GEX_data_filtered_logCPM.csv",
    index_col=0
)

In [6]:
pdo_gex = pd.read_csv(
    "../../integrated_data/pdo_logCPM.csv",
    index_col=0
)

In [7]:
pdo_gex = pdo_gex.loc[
    :,pdo_gex.columns.intersection(gdsc_gex.columns)]
pdo_gex = pdo_gex.reindex(columns=gdsc_gex.columns, fill_value=0.0)
pdo_gex.shape

(54, 2083)

In [8]:
pdo_dr = pd.read_csv(
    "../../integrated_data/dose_response_pdo.csv"
)

pdo_dr["Y_TRUE"] = (
    pdo_dr.groupby("Line")["LogIC50"]
    .transform(lambda x: stats.zscore(x, nan_policy="omit"))
)

pdo_dr = pdo_dr.loc[pdo_dr.seen_before=="yes",:]

In [9]:
smiles = pd.read_csv(
    "../../integrated_data/drug_smiles_pdo.tsv",
    index_col=0,
    sep="\t"
)

In [10]:
smiles.head()

,smiles
Entinostat,C1=CC=C(C(=C1)N)NC(=O)C2=CC=C(C=C2)CNC(=O)OCC3...
SB216763,CN1C=C(C2=CC=CC=C21)C3=C(C(=O)NC3=O)C4=C(C=C(C...
Taselisib,CC1=NN(C(=N1)C2=CN3CCOC4=C(C3=N2)C=CC(=C4)C5=C...
Axitinib,CNC(=O)C1=CC=CC=C1SC2=CC3=C(C=C2)C(=NN3)C=CC4=...
Camptothecin,CCC1(C2=C(COC1=O)C(=O)N3CC4=CC5=CC=CC=C5N=C4C3...


In [11]:
root = "../../cross_validation"
cancer_type = "pancancer"
experiment = "NBS_cells"
ckpt_dir = sorted(Path(f"cv/{cancer_type}_NBS_cells/").glob("version_*"))
n=10

per_drug_pcc_cv = {}
per_line_pcc_cv = {}

per_drug_precision_cv = {}
per_cell_precision_cv = {}

per_drug_ndcg_cv = {}
per_cell_ndcg_cv = {}

for f, ckpt in zip(range(n), ckpt_dir):
    fold_name = f"fold_{f}"
    ckpt_file = list((ckpt/"checkpoints").glob("*.ckpt"))[0]
    paccmann = Module_training_paccmann.load_from_checkpoint(
        ckpt_file,
        map_location="cuda",
        model=model,
        params = params
    )
    
    data_module = CVDataModule(
            root = root,
            type = cancer_type,
            experiment = experiment,
            fold_n = f,
            GEX_path = "../../paccmann_GEX_data_filtered_logCPM.csv",
            SMILES_path = "../../drug_smiles.tsv",
            SMILES_language = smiles_language,
            batch_size = params["batch_size"],
            num_workers = 22
        )

    data_module.setup()
    #train_mean = torch.tensor(data_module.gex_mean.to_numpy(), dtype=torch.float32)
    #train_std = torch.tensor(data_module.gex_std.to_numpy(), dtype=torch.float32)

    train_mean = data_module.gex_mean
    train_std = data_module.gex_std

    ########################################
    #
    # SUBSETTING TO PREDICTING SETS
    #
    ########################################
    gex = pdo_gex.loc[pdo_dr.Line,:]
    gex = (gex - train_mean)/train_std
    
    
    sm = smiles.loc[pdo_dr.Drug, :]
    sm = list(
        map(smiles_language.smiles_to_token_indexes, sm["smiles"].tolist())
    )
    
    #pdo_dr["Y_TRUE"] = (pdo_dr.LogIC50 - data_module.ic50_mean) / data_module.ic50_std

    test_ds = RegressionDataset(gex, sm, pdo_dr["Y_TRUE"])
    test_dl = DataLoader(
            test_ds,
            batch_size = params["batch_size"],
            shuffle = False,
            num_workers = 22
        )

    
    # Prediction
    trainer = L.Trainer(accelerator="auto")
    y_hat = trainer.predict(
        paccmann,
        dataloaders=test_dl
    )

    y_hat = torch.cat(y_hat).double().numpy()

    pdo_dr["Y_HAT"] = y_hat

    # Computing metrics
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_drug_pcc_cv[fold_name] = [per_drug_pcc["PCC"].median()]

    per_line_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_line_pcc_cv[fold_name] = [per_line_pcc["PCC"].median()]

    ##############################
    # PRECISION
    ##############################
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_drug_precision_cv[fold_name] = [per_drug_pcc["precision@q25"].median()]

    per_cell_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_cell_precision_cv[fold_name] = [per_cell_pcc["precision@q25"].median()]

    ##############################
    # NDCG
    ##############################
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_drug_ndcg_cv[fold_name] = [per_drug_pcc["ndcg@q25"].median()]

    per_cell_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_cell_ndcg_cv[fold_name] = [per_cell_pcc["ndcg@q25"].median()]


    


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/pancancer_NBS_cells/version_0/checkpoints/epoch=9-step=9670.ckpt


/home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/venv_paccmann/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/chp14eu/drug_repurposing/benchmarking/paccmann ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA L40') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set

DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_52/hparams.yaml
Predicting DataLoader 0: 100%|████████████████████████████████████████████| 9/9 [00:01<00:00,  5.45it/s]
DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/pancancer_NBS_cells/version_1/checkpoints/epoch=9-step=9660.ckpt


/tmp/ipykernel_429487/2928180313.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_429487/2928180313.py:93: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_429487/2928180313.py:104: FutureWarning: Da

DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_53/hparams.yaml
Predicting DataLoader 0: 100%|████████████████████████████████████████████| 9/9 [00:00<00:00, 22.01it/s]


...

/tmp/ipykernel_429487/2928180313.py:85: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_429487/2928180313.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_429487/2928180313.py:93: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_429487/2928180313.py:93: FutureWarning: DataFrameGro

DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/pancancer_NBS_cells/version_9/checkpoints/epoch=9-step=9680.ckpt


/tmp/ipykernel_429487/2928180313.py:134: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
/home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/venv_paccmann/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/chp14eu/drug_repurposing/benchmarking/paccmann ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, w

DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_61/hparams.yaml
Predicting DataLoader 0: 100%|████████████████████████████████████████████| 9/9 [00:00<00:00, 21.72it/s]


/tmp/ipykernel_429487/2928180313.py:85: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_429487/2928180313.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_429487/2928180313.py:93: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_429487/2928180313.py:93: FutureWarning: DataFrameGro

In [12]:
per_drug_pcc_cv = pd.DataFrame(per_drug_pcc_cv)
per_drug_pcc_cv.index = ["Paccmann"]
per_drug_pcc_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.024452,0.06657,-0.002906,-0.019384,0.02174,0.032472,0.03949,-0.00222,0.017618,-0.008773


In [24]:
M = np.median(per_drug_pcc_cv)
sem = stats.sem(per_drug_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.019678807967190418, low=0.0011555855877466055, high=0.03820203034663423


In [25]:
per_line_pcc_cv = pd.DataFrame(per_line_pcc_cv)
per_line_pcc_cv.index = ["Paccmann"]
per_line_pcc_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.45298,0.572077,0.430812,0.326795,0.282844,0.347954,0.310039,0.355408,0.213075,0.347155


In [26]:
M = np.median(per_line_pcc_cv)
sem = stats.sem(per_line_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.3475543924655763, low=0.2760374855866076, high=0.41907129934454496


In [27]:
per_drug_pcc_cv.to_csv(
    f"cv/pancancer_NBS_cells/predictions/pdo_fixed-drug_CV.csv"
)

In [28]:
per_line_pcc_cv.to_csv(
    f"cv/pancancer_NBS_cells/predictions/pdo_fixed-line_CV.csv"
)

In [13]:
per_drug_precision_cv = pd.DataFrame(per_drug_precision_cv)
per_drug_precision_cv.index = ["Paccmann"]
per_drug_precision_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.333333,0.333333,0.333333,0.333333,0.333333,0.316667,0.333333,0.25,0.333333,0.267857


In [14]:
per_drug_precision_cv.to_csv(
    f"cv/pancancer_NBS_cells/predictions/precision_pdo_fixed-drug_CV.csv"
)

In [15]:
per_drug_ndcg_cv = pd.DataFrame(per_drug_ndcg_cv)
per_drug_ndcg_cv.index = ["Paccmann"]
per_drug_ndcg_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.053019,0.0,0.0,0.0,0.0,0.0,0.0,0.012689,0.031354,0.0


In [16]:
per_drug_ndcg_cv.to_csv(
    f"cv/pancancer_NBS_cells/predictions/ndcg_pdo_fixed-drug_CV.csv"
)

In [17]:
per_cell_precision_cv = pd.DataFrame(per_cell_precision_cv)
per_cell_precision_cv.index = ["Paccmann"]
per_cell_precision_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.555556,0.277778,0.5,0.388889,0.277778,0.375,0.375,0.444444,0.555556,0.444444


In [18]:
per_cell_precision_cv.to_csv(
    "cv/pancancer_NBS_cells/predictions/precision_pdo_fixed-line_CV.csv"
)

In [19]:
per_cell_ndcg_cv = pd.DataFrame(per_cell_ndcg_cv)
per_cell_ndcg_cv.index = ["Paccmann"]
per_cell_ndcg_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.662268,0.169084,0.477422,0.424746,0.089658,0.462213,0.491289,0.380373,0.622471,0.40892


In [23]:
per_cell_ndcg_cv.to_csv(
    "cv/pancancer_NBS_cells/predictions/ndcg_pdo_fixed-line_CV.csv"
)

In [29]:
root = "../../cross_validation"
cancer_type = "pancancer"
experiment = "NBS_cells"
ckpt_dir = sorted(Path(f"cv/{cancer_type}_NBS_cells/").glob("version_*"))
n=10

per_cancer_pcc_cv = {}
per_lab_pcc_cv = {}

for f, ckpt in zip(range(n), ckpt_dir):
    fold_name = f"fold_{f}"
    ckpt_file = list((ckpt/"checkpoints").glob("*.ckpt"))[0]
    paccmann = Module_training_paccmann.load_from_checkpoint(
        ckpt_file,
        map_location="cuda",
        model=model,
        params = params
    )
    
    data_module = CVDataModule(
            root = root,
            type = cancer_type,
            experiment = experiment,
            fold_n = f,
            GEX_path = "../../paccmann_GEX_data_filtered_logCPM.csv",
            SMILES_path = "../../drug_smiles.tsv",
            SMILES_language = smiles_language,
            batch_size = params["batch_size"],
            num_workers = 22
        )

    data_module.setup()
    #train_mean = torch.tensor(data_module.gex_mean.to_numpy(), dtype=torch.float32)
    #train_std = torch.tensor(data_module.gex_std.to_numpy(), dtype=torch.float32)

    train_mean = data_module.gex_mean
    train_std = data_module.gex_std

    ########################################
    #
    # SUBSETTING TO PREDICTING SETS
    #
    ########################################
    gex = pdo_gex.loc[pdo_dr.Line,:]
    gex = (gex - train_mean)/train_std
    
    
    sm = smiles.loc[pdo_dr.Drug, :]
    sm = list(
        map(smiles_language.smiles_to_token_indexes, sm["smiles"].tolist())
    )
    
    pdo_dr["Y_TRUE"] = (pdo_dr.LogIC50 - data_module.ic50_mean) / data_module.ic50_std

    test_ds = RegressionDataset(gex, sm, pdo_dr["Y_TRUE"])
    test_dl = DataLoader(
            test_ds,
            batch_size = params["batch_size"],
            shuffle = False,
            num_workers = 22
        )

    
    # Prediction
    trainer = L.Trainer(accelerator="auto")
    y_hat = trainer.predict(
        paccmann,
        dataloaders=test_dl
    )

    y_hat = torch.cat(y_hat).double().numpy()

    pdo_dr["Y_HAT"] = y_hat

    cancer_type = (
        pdo_dr[["Line","TCGA_DESC"]]
        .drop_duplicates()
    )

    labs = (
        pdo_dr[["Line", "Lab"]]
        .drop_duplicates()
    )


    # Computing metrics
    per_cancer_pcc = (pdo_dr.groupby("TCGA_DESC")
                    .filter(lambda x: len(x) >=2)
                    .groupby("TCGA_DESC")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_cancer_pcc = (per_cancer_pcc
                      .merge(cancer_type, on="TCGA_DESC", how="left")
                      .dropna(subset=["TCGA_DESC"])
                      .groupby("TCGA_DESC")["PCC"]
                      .median()
                     )

    per_cancer_pcc_cv[fold_name] = per_cancer_pcc

    per_lab_pcc = (pdo_dr.groupby("Lab")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Lab")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_lab_pcc = (per_lab_pcc
                   .merge(labs, on="Lab", how="left")
                   .dropna(subset=["Lab"])
                   .groupby("Lab")["PCC"]
                   .median()
                   )

    per_lab_pcc_cv[fold_name] = per_lab_pcc


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/pancancer_NBS_cells/version_0/checkpoints/epoch=9-step=9670.ckpt


/home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/venv_paccmann/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/chp14eu/drug_repurposing/benchmarking/paccmann ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_42/hparams.yaml
Predicting DataLoader 0: 100%|█████████████████████████████████████████████| 9/9 [00:00<00:00, 20.90it/s]
DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/pancancer_NBS_cells/version_1/checkpoints/epoch=9-step=9660.ckpt


/tmp/ipykernel_3714700/2416713219.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_3714700/2416713219.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/home/chp14eu/drug_repurposing/benchmarking/paccmann_pr

DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_43/hparams.yaml
Predicting DataLoader 0: 100%|█████████████████████████████████████████████| 9/9 [00:00<00:00, 21.06it/s]
DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/pancancer_NBS_cells/version_2/checkpoints/epoch=9-step=9670.ckpt


...

/tmp/ipykernel_3714700/2416713219.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_3714700/2416713219.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/home/chp14eu/drug_repurposing/benchmarking/paccmann_pr

DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_50/hparams.yaml
Predicting DataLoader 0: 100%|█████████████████████████████████████████████| 9/9 [00:00<00:00, 21.14it/s]
DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/cv/pancancer_NBS_cells/version_9/checkpoints/epoch=9-step=9680.ckpt


/tmp/ipykernel_3714700/2416713219.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_3714700/2416713219.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/home/chp14eu/drug_repurposing/benchmarking/paccmann_pr

DEBUG:fsspec.local:open file: /home/chp14eu/drug_repurposing/benchmarking/paccmann_predictor/lightning_logs/version_51/hparams.yaml
Predicting DataLoader 0: 100%|█████████████████████████████████████████████| 9/9 [00:00<00:00, 20.86it/s]


/tmp/ipykernel_3714700/2416713219.py:90: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_3714700/2416713219.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


In [32]:
cancer_type = (pd.DataFrame(per_cancer_pcc_cv)
               .groupby(level=0)
               .median()
               .median(axis=1)
               .rename_axis("")
               .T
)
cancer_type.name = "Paccmann"
cancer_type


BLCA    0.202282
COAD    0.320584
HNSC    0.143083
PDAC    0.599227
Name: Paccmann, dtype: float64

In [33]:
cancer_type.to_csv(
    "cv/pancancer_NBS_cells/predictions/pdo_fixed-cancer_CV.csv"
)